In [ ]:
import numpy as np
import pandas as pd

def risk_dashboard(returns: pd.Series, rf: float = 0.0, periods: int = 252, target: float = 0.0) -> dict:
    """
    returns: pandas Series of periodic returns (simple or log). Indexed by date recommended.
    rf: risk-free rate per period (daily if returns are daily). For now you can keep rf=0.
    periods: periods per year (252 for daily trading days).
    target: target/threshold return for downside risk (0 by default).
    
    Notes:
    - Kurtosis returned is EXCESS kurtosis if using pandas/scipy default fisher=True behavior.
      (Normal distribution => 0 excess kurtosis.)
    - Max drawdown uses cumulative growth:
        * For log returns: exp(cumsum)
        * For simple returns: (1+returns).cumprod
      We detect log vs simple conservatively; you can override by passing log/simple explicitly if you want later.
      Kurtosis is excess kurtosis.
    """
    if returns is None:
        raise ValueError("returns is None")
        
    # Ensure Series, drop NaNs
    if isinstance(returns, pd.DataFrame):
        returns = returns.iloc[:, 0]
    returns = returns.dropna()

    if len(returns) < 5:
        raise ValueError("Not enough return observations to compute dashboard reliably.")

    # Mean and volatility (per period)
    mu = float(returns.mean())
    sigma = float(returns.std(ddof=1))

    # Downside returns and downside deviation (per period)
    downside = returns[returns < target]
    downside_sigma = float(downside.std(ddof=1)) if len(downside) > 1 else np.nan

    # Annualization
    mu_annual = mu * periods
    vol_annual = sigma * np.sqrt(periods)
    downside_vol_annual = downside_sigma * np.sqrt(periods) if np.isfinite(downside_sigma) else np.nan

    # Sharpe & Sortino (annualized via daily->annual scaling)
    # Excess return per period: (mu - rf)
    # Annualized Sharpe: (mu-rf)/sigma * sqrt(periods)
    sharpe_annual = ((mu - rf) / sigma) * np.sqrt(periods) if sigma != 0 else np.nan
    sortino_annual = ((mu - rf) / downside_sigma) * np.sqrt(periods) if (np.isfinite(downside_sigma) and downside_sigma != 0) else np.nan

    # Distribution shape
    skew = float(returns.skew())
    kurt_excess = float(returns.kurt())  # pandas -> excess kurtosis (normal => 0)

    # Max Drawdown
    # Decide cumulative growth:
    # - If returns ever less than -1, it's almost certainly log returns (simple returns can't be < -1).
    # - Otherwise: if typical magnitudes are small, both are plausible; we'll treat as log if values look like log returns.
    is_log_like = (returns.min() < -1.0)
    if is_log_like:
        cum = np.exp(returns.cumsum())
    else:
        # If returns are log returns, (1+returns) is slightly wrong but close for small values.
        # For your pipeline, you're using log returns, so you can optionally force log by setting is_log_like=True.
        cum = (1.0 + returns).cumprod()

    running_max = cum.cummax()
    drawdown = (cum - running_max) / running_max
    max_drawdown = float(drawdown.min())

    summary = {
        "n_obs": int(len(returns)),
        "mean_daily": mu,
        "mean_annual": mu_annual,
        "vol_annual": vol_annual,
        "downside_vol_annual": downside_vol_annual,
        "sharpe_annual": sharpe_annual,
        "sortino_annual": sortino_annual,
        "skew": skew,
        "kurtosis_excess": kurt_excess,
        "max_drawdown": max_drawdown,
    }
    return summary

In [2]:
risk_dashboard(tsla_returns)
risk_dashboard(spy_returns)
risk_dashboard(strategy_returns)

NameError: name 'tsla_returns' is not defined